In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_M05.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 0.779981926930981, 'n_it': 0.3750465847460756}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=3)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}
            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 0 =========================================
18.115876767562312

Trial 1 =========================================
18.34435490115575

Trial 2 =========================================
18.028438914561598

Trial 3 =========================================
17.613744029467018

Trial 4 =========================================
18.299652119644485

Trial 5 =========================================
16.553983103253373

Trial 6 =========================================
17.929099808640988

Trial 7 =========================================
18.375004079312315

Trial 8 =========================================
17.616772876105045

Trial 9 =========================================
17.56384544559333

Trial 10 =========================================
18.057462703810412

Trial 11 =========================================
17.945418440655235

Trial 12 =========================================
18.341565320339733

Trial 13 =========================================
18.07366810090958

Trial 14 ==========

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 68 =========================================
17.36777593564326

Trial 69 =========================================
18.36449979973679

Trial 70 =========================================
17.863030177993288

Trial 71 =========================================
18.014893318316116

Trial 72 =========================================
18.23977443600455

Trial 73 =========================================
18.326725914838857

Trial 74 =========================================
17.62307397425719

Trial 75 =========================================
16.68628686749929

Trial 76 =========================================
17.75676656973409

Trial 77 =========================================
16.869583704588944

Trial 78 =========================================
17.96114264006107

Trial 79 =========================================
17.589679053304895

Trial 80 =========================================
17.823768418932445

Trial 81 =========================================
18.108943216176776

Trial 82 ====

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.512627514841864
Avg = 17.767200007295546
Std = 0.8363961908223148


In [7]:
print(y_max_arr.tolist())

[18.115876767562312, 18.34435490115575, 18.028438914561598, 17.613744029467018, 18.299652119644485, 16.553983103253373, 17.929099808640988, 18.375004079312315, 17.616772876105045, 17.56384544559333, 18.057462703810412, 17.945418440655235, 18.341565320339733, 18.07366810090958, 15.97951160998988, 18.24360549670793, 18.343072613319205, 17.741350526785368, 18.087213633087273, 18.27610891987195, 17.411963712666434, 18.090574610713347, 18.41188754744107, 17.953787433030655, 17.94259875969198, 17.557061729450286, 17.864914973283646, 17.886357121429167, 18.288789515789645, 17.777362813668663, 17.94692537763619, 17.421906932157896, 17.893601502757903, 17.819939205943868, 13.752858017827416, 18.182973148663688, 18.467749717639848, 18.249514152897287, 17.676169190246988, 17.779120089325126, 17.89080767066723, 18.092695094682465, 18.239289207464353, 18.44368463050159, 18.053753765493667, 17.985628308461408, 18.036945028967548, 18.10660305771131, 18.307569609579588, 14.394946732608169, 18.30029743

In [8]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M05/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_M05/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    17.242860
1    17.146810
2    17.907134
3    17.218415
4    18.460924
..         ...
295  17.820441
296  18.300400
297  18.310506
298  13.777737
299  17.973181

[300 rows x 1 columns]
